### <mark> 1단계: 데이터 불러오기 : MNIST<mark>

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
transform = transforms.ToTensor()

In [3]:
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 46.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 54.5MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 118MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.10MB/s]


### <mark> 2단계: SimpleCNN 클래스 </mark>

In [4]:
from torch import nn

In [5]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
    self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
    self.pool = nn.MaxPool2d(2,2)
    self.flatten = nn.Flatten() #데이터를 한 줄로 만들기
    self.fc1 = nn.Linear(128 * 3 * 3, 256)
    self.fc2 = nn.Linear(256, 10)

  def forward(self,x):
    x = torch.relu(self.conv1(x))
    x = self.pool(x)

    x = torch.relu(self.conv2(x))
    x = self.pool(x)

    x = torch.relu(self.conv3(x))
    x = self.pool(x)

    x = self.flatten(x)
    x = torch.relu(self.fc1(x))
    x = self.fc2(x)

    return x

In [6]:
model = SimpleCNN()

print(model)

SimpleCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=1152, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)


In [7]:
images, labels = next(iter(train_loader))
output = model(images)
print(output.shape)

torch.Size([64, 10])


In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [9]:
for epoch in range(5):
  model.train()
  running_loss = 0.0

  for inputs, labels in train_loader:
    optimizer.zero_grad()

    outputs = model(inputs)
    loss = loss_fn(outputs, labels)

    loss.backward()

    optimizer.step()

    running_loss += loss.item()

print(f'epoch : {epoch+1}, Loss : {running_loss/len(train_loader)}')

epoch : 5, Loss : 0.019901715621800948


In [10]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
  for test_inputs, test_labels in test_loader:
    test_outputs = model(test_inputs)
    _, predicted = torch.max(test_outputs, 1)
    total += test_labels.size(0)
    correct += (predicted == test_labels).sum().item()

In [11]:
accuracy = correct/total
print(f'accuracy : {accuracy * 100}%')

accuracy : 98.76%


### <font color='red'> claude와 대화 </font>
https://claude.ai/share/fe439548-92f0-49f7-89c1-e5a6ece64530